# Joint modeling of annual precipitation maxima over several durations for the construction of intensity-duration-frequency curves

Paul Mathivon$^{1,2}$, Christian Genest$^{3}$ and Jonathan Jalbert$^{1}$

$^1$Polytechnique Montréal\
$^2$ENGIE\
$^3$McGill University

June 13, 2024

This program reproduces the results within the paper of Mathivon *et al.* (2024+).

The notebook runs with a Julia 1.10 kernel and uses these package versions:
- CSV v0.10.14
- Combinatorics v1.0.2
- DataFrames v1.6.1
- Distributions v0.25.109
- Extremes v1.0.2
- IDFCurves v0.2.0
- Gadfly v1.4.0
- StatsBase v0.33.21


### Reference

Mathivon, P., Genest, C. and Jalbert, J. (2024). Joint modeling of annual precipitation maxima over several durations for the construction of intensity-duration-frequency curves. *Submitted to Water Resources Research*.


In [ ]:
using Cairo, Combinatorics, CSV, DataFrames, Distributions, Extremes, Fontconfig, Gadfly
using IDFCurves, LinearAlgebra, StatsBase

# Data

## Data loading

In [ ]:
df = IDFCurves.dataset("702S006")
# df= CSV.read("6158731.csv", missingstring="-99.9", DataFrame)
first(df,5)

In [ ]:
tags = names(df)[2:10]
durations = [1/12, 1/6, 1/4, 1/2, 1, 2, 6, 12, 24]
duration_dict = Dict(zip(tags, durations))

In [ ]:
data = IDFdata(df, "Year", duration_dict)

## Exploratory analysis

### Correlation vs durations

In [ ]:
fig = Gadfly.Plot[]

p = plot(df, x="1h", y="5min", Theme(default_color="black"))
push!(fig, p)
# draw(PDF("5minvs1h.pdf", 10cm, 8cm), p)

p = plot(df, x="1h", y="15min", Theme(default_color="black"))
push!(fig, p)
# draw(PDF("15minvs1h.pdf", 10cm, 8cm), p)

p = plot(df, x="1h", y="30min", Theme(default_color="black"))
push!(fig, p)
# draw(PDF("30minvs1h.pdf", 10cm, 8cm), p)

p = plot(df, x="1h", y="2h", Theme(default_color="black"))
push!(fig, p)
# draw(PDF("2hvs1h.pdf", 10cm, 8cm), p)

p = plot(df, x="1h", y="6h", Theme(default_color="black"))
push!(fig, p)
# draw(PDF("6hvs1h.pdf", 10cm, 8cm), p)

p = plot(df, x="1h", y="24h", Theme(default_color="black"))
push!(fig, p)
# draw(PDF("24hvs1h.pdf", 10cm, 8cm), p)

Gadfly.set_default_plot_size(30cm, 20cm)
gridstack([fig[1] fig[2] fig[3];
    fig[4] fig[5] fig[6]])

In [ ]:
df_kendall = DataFrame(Distance = Float64[], Kendall = Float64[])

for c in combinations(gettag(data),2)
    
    d₁ = getduration(data, c[1])
    d₂ = getduration(data, c[2])
    
    h = IDFCurves.logdist(d₁, d₂)
    
    y₁ = getdata(data, c[1])
    y₂ = getdata(data, c[2])
    
    τ = corkendall(y₁, y₂)
    
    push!(df_kendall, [h, τ])
    
end

df_kendall.Kendall_sin = sin.(π/2*df_kendall.Kendall);

In [ ]:
Gadfly.set_default_plot_size(10cm, 8cm)

fig = plot(df_kendall, x=:Distance, y=:Kendall, Theme(default_color="black"),
    Guide.xlabel("h(d, d')"), Guide.ylabel("Kendall's τ"))

# draw(PDF("tau_vs_logratio.pdf", 10cm, 8cm), fig)

# Independent GEV

In [ ]:
fig = Gadfly.Plot[]

for d in gettag(data)
    
    y = getdata(data, d)
    fd = gevfit(y)
    p = Extremes.qqplotci(fd)   
    
    push!(fig, p)
    
#     figname = string("GEV_",d,".pdf")
#     draw(PDF(figname, 10cm, 8cm), p)
    
end


Gadfly.set_default_plot_size(30cm, 30cm)
gridstack([fig[1] fig[2] fig[3];
    fig[4] fig[5] fig[6];
    fig[7] fig[8] fig[9]])


# General scaling model

## Model fit

In [ ]:
dGEVmodel = IDFCurves.fit_mle(GeneralScaling, data, 1, [20, 5, .04, .76, .07])

## Parameter Wald confidence intervals 

In [ ]:
H = IDFCurves.hessian(dGEVmodel, data)

Σ = inv(H)

W = Normal.(collect(params(dGEVmodel)), sqrt.(diag(Σ)))

bounds = round.(quantile.(W, [.025 .975]), digits=4)

## Scaling location parameters vs marginal location parameters 

In [ ]:
d₀ = duration(dGEVmodel)
θ̂ = collect(params(dGEVmodel))
H = IDFCurves.hessian(dGEVmodel, data)

df_model = DataFrame(Duration = Float64[], Estimate = Float64[], lbound = Float64[], ubound = Float64[])

model(θ::AbstractVector{<:Real}) = GeneralScaling(duration(dGEVmodel), θ...)

for d in getduration.(data, gettag(data))
   
    μ̂ = location(getdistribution(model(θ̂), d))
    
    g(θ::AbstractVector{<:Real}) = location(getdistribution(model(θ), d))
        
    v = Extremes.delta(g, θ̂, H)
    
    lbound, rbound = quantile(Normal(μ̂, sqrt(v)), [.025, .975])
    
    push!(df_model, [d, μ̂, lbound, rbound])
    
end

df_model


In [ ]:
df_empirical = DataFrame(Duration = Float64[], Estimate = Float64[], lbound = Float64[], ubound = Float64[])

for tag in gettag(data)
   
    d = getduration(data, tag)
    
    fd = gevfit(getdata(data, tag))
        
    c = cint(fd)
    
    push!(df_empirical, [d, location(fd)[], first(c[1]), last(c[1])])
    
end

df_empirical

In [ ]:
xlabel(x) = gettag(data, exp(x))
ylabel(y) = string(Int(round(exp(y))))

empirical = layer(df_empirical, x=:Duration, ymin=:lbound, ymax=:ubound, Geom.errorbar)
modeled = layer(df_model, x=:Duration, y=:Estimate, Geom.line,
    ymin=:lbound, ymax=:ubound, Geom.ribbon)

Gadfly.set_default_plot_size(10cm, 8cm)

fig = plot(empirical, modeled, 
    Guide.xticks(ticks = log.(durations)), Scale.x_log(labels=xlabel),
    Guide.yticks(ticks = log.([1, 2, 5, 10, 20, 50, 100])), Scale.y_log(labels=ylabel),
    Guide.ylabel("Location"),
    Theme(default_color="black", lowlight_color=c->"lightgray"))

# draw(PDF("dGEV_location.pdf", 10cm, 8cm), fig)


## Scaling scale parameters vs marginal scale parameters 

In [ ]:
d₀ = duration(dGEVmodel)
θ̂ = collect(params(dGEVmodel))
H = IDFCurves.hessian(dGEVmodel, data)

df_model = DataFrame(Duration = Float64[], Estimate = Float64[], lbound = Float64[], ubound = Float64[])

model(θ::AbstractVector{<:Real}) = GeneralScaling(duration(dGEVmodel), θ...)

for d in getduration.(data, gettag(data))
   
    σ̂ = Extremes.scale(getdistribution(model(θ̂), d))
    
    g(θ::AbstractVector{<:Real}) = Extremes.scale(getdistribution(model(θ), d))
        
    v = Extremes.delta(g, θ̂, H)
    
    lbound, rbound = quantile(Normal(σ̂, sqrt(v)), [.025, .975])
    
    push!(df_model, [d, σ̂, lbound, rbound])
    
end

df_model

In [ ]:
df_empirical = DataFrame(Duration = Float64[], Estimate = Float64[], lbound = Float64[], ubound = Float64[])

for tag in gettag(data)
   
    d = getduration(data, tag)
    
    fd = gevfit(getdata(data, tag))
        
    c = cint(fd)
    
    push!(df_empirical, [d, Extremes.scale(fd)[], exp(first(c[2])), exp(last(c[2]))])
    
end

df_empirical

In [ ]:
empirical = layer(df_empirical, x=:Duration, ymin=:lbound, ymax=:ubound, Geom.errorbar)
modeled = layer(df_model, x=:Duration, y=:Estimate, Geom.line,
    ymin=:lbound, ymax=:ubound, Geom.ribbon)

Gadfly.set_default_plot_size(10cm, 8cm)

fig = plot(empirical, modeled, 
    Guide.xticks(ticks = log.(durations)), Scale.x_log(labels=xlabel),
    Guide.yticks(ticks = log.([1, 2, 5, 10, 20, 40])), Scale.y_log(labels=ylabel),
    Guide.ylabel("Scale"),
    Theme(default_color="black", lowlight_color=c->"lightgray"))

# draw(PDF("dGEV_scale.pdf", 10cm, 8cm), fig)

## Scaling shape parameters vs marginal shape parameters 

In [ ]:
d₀ = duration(dGEVmodel)
θ̂ = collect(params(dGEVmodel))
H = IDFCurves.hessian(dGEVmodel, data)

df_model = DataFrame(Duration = Float64[], Estimate = Float64[], lbound = Float64[], ubound = Float64[])

model(θ::AbstractVector{<:Real}) = GeneralScaling(duration(dGEVmodel), θ...)

for d in getduration.(data, gettag(data))
   
    ξ̂ = Extremes.shape(getdistribution(model(θ̂), d))
    
    g(θ::AbstractVector{<:Real}) = Extremes.shape(getdistribution(model(θ), d))
        
    v = Extremes.delta(g, θ̂, H)
    
    lbound, rbound = quantile(Normal(ξ̂, sqrt(v)), [.025, .975])
    
    push!(df_model, [d, ξ̂, lbound, rbound])
    
end

df_model

In [ ]:
df_empirical = DataFrame(Duration = Float64[], Estimate = Float64[], lbound = Float64[], ubound = Float64[])

for tag in gettag(data)
   
    d = getduration(data, tag)
    
    fd = gevfit(getdata(data, tag))
        
    c = cint(fd)
    
    push!(df_empirical, [d, Extremes.shape(fd)[], first(c[3]), last(c[3])])
    
end

df_empirical

In [ ]:
empirical = layer(df_empirical, x=:Duration, ymin=:lbound, ymax=:ubound, Geom.errorbar)
modeled = layer(df_model, x=:Duration, y=:Estimate, Geom.line,
    ymin=:lbound, ymax=:ubound, Geom.ribbon)

Gadfly.set_default_plot_size(10cm, 8cm)

fig = plot(empirical, modeled, 
    Guide.xticks(ticks = log.(durations)), Scale.x_log(labels=xlabel),
    Guide.ylabel("Shape"),
    Theme(default_color="black", lowlight_color=c->"lightgray"))

# draw(PDF("dGEV_shape.pdf", 10cm, 8cm), fig)


## Fit of the General Scaling model

In [ ]:
fig = Gadfly.Plot[]

for d in gettag(data)
    
    p = IDFCurves.qqplotci(dGEVmodel, data, getduration(data, d))
        
    push!(fig, p)
        
#     figname = string("dGEV_",d,".pdf")
#     draw(PDF(figname, 10cm, 8cm), p)
    
end

Gadfly.set_default_plot_size(30cm, 30cm)
gridstack([fig[1] fig[2] fig[3];
    fig[4] fig[5] fig[6];
    fig[7] fig[8] fig[9]])

# General scaling model with a copula

## Likelihood as a function of the t copula degrees of freedom

In [ ]:
initialvalues = [20, 5, .04, .76, .07, 1., 2.]

df = DataFrame(ν = Int64[], loglikelihood = Float64[])

for ν in 5:5:30

    ddGEVmodel = IDFCurves.fit_mle(DependentScalingModel{GeneralScaling, MaternCorrelationStructure, TCopula{ν}},
        data, 1, initialvalues)

    ℓ = IDFCurves.loglikelihood(ddGEVmodel, data)
    
    push!(df, [ν, ℓ])
    
end

df

In [ ]:
Gadfly.set_default_plot_size(10cm, 8cm)

fig = plot(df, x=:ν, y=:loglikelihood, Geom.point, Geom.line, 
    Theme(default_color="black", lowlight_color=c->"lightgray"),
    Guide.xticks(ticks=5:5:30), Guide.xlabel("Degrees of freedom"),
    Guide.ylabel("Loglikelihood"))

# draw(PDF("TCopula_loglike.pdf", 10cm, 8cm), fig)

## General Scaling model with a Gaussian copula

In [ ]:
initialvalues = [20, 5, .04, .76, .07, 1., 2.]

ddGEVmodel = IDFCurves.fit_mle(DependentScalingModel{GeneralScaling, MaternCorrelationStructure, GaussianCopula}, data, 1, initialvalues)

### Parameter Wald confidence intervals 

In [ ]:
H = IDFCurves.hessian(ddGEVmodel, data)

Σ = inv(H)

W = Normal.(collect(params(ddGEVmodel)), sqrt.(diag(Σ)))

bounds = round.(quantile.(W, [.025 .975]), digits=4)

### Correlation model fit

In [ ]:
durations = getduration.(data, gettag(data))
h = IDFCurves.logdist(durations)

c = IDFCurves.getcorrelogram(ddGEVmodel)

In [ ]:
ρ(h::Real) = cor(c, h)

model_cor = layer(ρ, 0, 6, Theme(default_color="black"))
sample = layer(df_kendall, x=:Distance, y=:Kendall_sin, Theme(default_color="black"))

fig = plot(model_cor, sample, Coord.cartesian(ymax=1), Guide.xlabel("h"), Guide.ylabel("Σ"))
            
# draw(PDF("corr_vs_h_gaussiancopula.pdf", 10cm, 8cm), fig)

### Model fit

In [ ]:
fig = Gadfly.Plot[]

for d in gettag(data)
    
    p = IDFCurves.qqplotci(ddGEVmodel, data, getduration(data, d))
        
    push!(fig, p)
        
#     figname = string("ddGEV_",d,".pdf")
#     draw(PDF(figname, 10cm, 8cm), p)
    
end

Gadfly.set_default_plot_size(30cm, 30cm)
gridstack([fig[1] fig[2] fig[3];
    fig[4] fig[5] fig[6];
    fig[7] fig[8] fig[9]])

# Model misspecification

In [ ]:
G = IDFCurves.godambe(dGEVmodel, data)

In [ ]:
for d in gettag(data)
    
    p = IDFCurves.qqplotci(dGEVmodel, data, getduration(data, d), G)
        
    figname = string("dGGEV_",d,".pdf")
    
#     draw(PDF(figname, 10cm, 8cm), p)
    
end

In [ ]:
fig = Gadfly.Plot[]

for d in gettag(data)
    
    p = IDFCurves.qqplotci(dGEVmodel, data, getduration(data, d), G)
        
    push!(fig, p)
        
#     figname = string("dGGEV_",d,".pdf")
#     draw(PDF(figname, 10cm, 8cm), p)
    
end

Gadfly.set_default_plot_size(30cm, 30cm)
gridstack([fig[1] fig[2] fig[3];
    fig[4] fig[5] fig[6];
    fig[7] fig[8] fig[9]])

# Toronto data

In [ ]:
data = CSV.read("6158731.CSV", DataFrame)